In [1]:

import pandas as pd
from pathlib import Path
from src.data_process.k_means_clasification import cluster_por_macrozona
df_path = Path(r"D:\ProyectoAnalisisElectrico\DiaPromedio\Periodos\2505_2604\2505_2604_mean_period_loc.parquet")

In [2]:
df = pd.read_parquet(df_path)

In [3]:
print(df.columns)
df.head()

Index(['clave', 'Zona', 'Hora', 'medida', 'CMg[CLP/KWh]', 'valorizado_CLP',
       'Calendario_Activo', 'RUT', 'rut_log', 'n_ruts', 'Razon_Social',
       'razon_social_log', 'n_razones_sociales', 'Nombre_Corto',
       'nombre_corto_log', 'n_nombres_cortos', 'nombre_barra',
       'nombre_barra_log', 'n_nombres_barra', 'tension', 'tension_log',
       'n_tensiones', 'tipo', 'period', 'medida_total', 'region', 'macrozona',
       'confianza', 'IA'],
      dtype='object')


,clave,Zona,Hora,medida,CMg[CLP/KWh],valorizado_CLP,Calendario_Activo,RUT,rut_log,n_ruts,...,tension,tension_log,n_tensiones,tipo,period,medida_total,region,macrozona,confianza,IA
0,$C$439,Norte,0,-16265.098843,66.273480,-1.068584e+06,111111111111,96.505.760-9,96.505.760-9,1,...,220,220,1,L,2505_2604,-374611.038665,Antofagasta,Norte Grande,100.0,False
1,$C$439,Norte,1,-16018.128504,65.684023,-1.044962e+06,111111111111,96.505.760-9,96.505.760-9,1,...,220,220,1,L,2505_2604,-374611.038665,Antofagasta,Norte Grande,100.0,False
2,$C$439,Norte,2,-15406.094948,66.415769,-1.018780e+06,111111111111,96.505.760-9,96.505.760-9,1,...,220,220,1,L,2505_2604,-374611.038665,Antofagasta,Norte Grande,100.0,False
3,$C$439,Norte,3,-15885.848474,66.879940,-1.053275e+06,111111111111,96.505.760-9,96.505.760-9,1,...,220,220,1,L,2505_2604,-374611.038665,Antofagasta,Norte Grande,100.0,False
4,$C$439,Norte,4,-15792.851028,66.719561,-1.045057e+06,111111111111,96.505.760-9,96.505.760-9,1,...,220,220,1,L,2505_2604,-374611.038665,Antofagasta,Norte Grande,100.0,False


In [ ]:
print("Calculando proporciones diarias...")

df["medida_porcentual"] = df["medida"] / df["medida_total"].replace(0, 1.0)

print("Iniciando segmentación DTW por macrozonas...\n")

df_centros, df_etiquetado = cluster_por_macrozona(
    df=df,
    value_col="medida_porcentual",  # <-- Usamos la nueva columna de porcentajes
    window=3,                       
    normalize=False,                # <-- IMPORTANTE: Lo dejamos en False
    k_max_global=10                 
)



Calculando proporciones diarias...
Iniciando segmentación DTW por macrozonas...


Macrozona: Norte Grande
 └─ Claves únicas (clientes/barras): 473
 └─ Total de registros: 11,352
Buscando codo DTW en rango 2-10...
✅ Codo DTW detectado en K=4

Macrozona: Centro Sur
 └─ Claves únicas (clientes/barras): 1,406
 └─ Total de registros: 33,744
Buscando codo DTW en rango 2-10...
✅ Codo DTW detectado en K=3

Macrozona: Norte Chico
 └─ Claves únicas (clientes/barras): 407
 └─ Total de registros: 9,768
Buscando codo DTW en rango 2-10...
✅ Codo DTW detectado en K=3

Macrozona: Centro
 └─ Claves únicas (clientes/barras): 3,553
 └─ Total de registros: 85,272
Buscando codo DTW en rango 2-10...
✅ Codo DTW detectado en K=5

Macrozona: Sur
 └─ Claves únicas (clientes/barras): 954
 └─ Total de registros: 22,920
Buscando codo DTW en rango 2-10...
✅ Codo DTW detectado en K=6

✅ PROCESO COMPLETADO EXITOSAMENTE CON DTW

--- VISTA PREVIA DE LOS CENTROIDES (PROPORCIONES) ---


,macrozona,id_cluster,clave_medoid,dtw_inertia,Hora,medida_porcentual
0,Norte Grande,0,36135733,46.471641,0,0.015050
1,Norte Grande,1,36140777,46.471641,0,0.035967
2,Norte Grande,2,02605927000RC,46.471641,0,0.000000
3,Norte Grande,3,15848400000RF,46.471641,0,0.042697
4,Norte Grande,0,36135733,46.471641,1,0.014876
5,Norte Grande,1,36140777,46.471641,1,0.035865
6,Norte Grande,2,02605927000RC,46.471641,1,0.000000
7,Norte Grande,3,15848400000RF,46.471641,1,0.042751
8,Norte Grande,0,36135733,46.471641,2,0.014943
9,Norte Grande,1,36140777,46.471641,2,0.034919


In [12]:
# 1. Definir la carpeta de destino
carpeta_salida = Path(r"D:\ProyectoAnalisisElectrico\DiaPromedio\Periodos\2505_2604")

# 2. Definir los nuevos nombres de los archivos con el sufijo _porcentual
ruta_centros_porcentual = carpeta_salida / "2505_2604_centroids_porcentual.parquet"
ruta_etiquetado_porcentual = carpeta_salida / "2505_2604_clustered_porcentual.parquet"

# 3. Guardar en formato Parquet
print("Guardando archivos en formato porcentual...")
df_centros.to_parquet(ruta_centros_porcentual, engine="pyarrow", compression="snappy")
df_etiquetado.to_parquet(ruta_etiquetado_porcentual, engine="pyarrow", compression="snappy")

print("--- EXPORTACIÓN EXITOSA ---")
print(f"✅ Centros porcentuales guardados en:\n   {ruta_centros_porcentual}")
print(f"✅ Datos etiquetados porcentuales guardados en:\n   {ruta_etiquetado_porcentual}")

Guardando archivos en formato porcentual...
--- EXPORTACIÓN EXITOSA ---
✅ Centros porcentuales guardados en:
   D:\ProyectoAnalisisElectrico\DiaPromedio\Periodos\2505_2604\2505_2604_centroids_porcentual.parquet
✅ Datos etiquetados porcentuales guardados en:
   D:\ProyectoAnalisisElectrico\DiaPromedio\Periodos\2505_2604\2505_2604_clustered_porcentual.parquet
